# Requirement 1 - Single Campaign

This notebook is the cleaned final notebook for Requirement 1. It is aligned with the current repository code and with the final exported Req1 figures stored in `outputs/req1/`.

The experiment studies a repeated first-price auction for a single campaign. The learner chooses a bid from a finite grid, observes stochastic outcomes, and receives utility only when the bid beats the highest competing bid. The key comparison is between:

- **UCB1**, a standard bandit learner that ignores the budget;
- **UCB-like budget-aware bidding**, which uses an LP to respect the per-round budget rate.


In [ ]:
from pathlib import Path
import pickle
import sys

import numpy as np
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "picklefiles"
OUTPUTS_DIR = ROOT / "outputs"

def load_pickle(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run the corresponding experiment first.")
    with path.open("rb") as f:
        return pickle.load(f)

def show_png(relative_path, width=900):
    path = OUTPUTS_DIR / relative_path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"**Missing plot:** `{path}`"))

def markdown_table(rows, columns, formats=None):
    formats = formats or {}
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in rows:
        vals = []
        for col in columns:
            value = row[col]
            fmt = formats.get(col)
            vals.append(fmt.format(value) if fmt else str(value))
        lines.append("| " + " | ".join(vals) + " |")
    display(Markdown("\n".join(lines)))


## Code Path and Parameters

The authoritative implementation for this requirement is `utils/run_req1.py`. The environment is `SingleCampaignEnv`, while the learners are `UCB1BiddingAgent` and `UCBLikeBiddingAgent`.


In [ ]:
from utils.environments import SingleCampaignEnv
from utils.experiments import compute_clairvoyant_single
from utils.run_req1 import (
    AVAILABLE_BIDS,
    BUDGET_GENEROUS,
    BUDGET_TIGHT,
    N_COMPETITORS,
    N_TRIALS,
    T,
    VALUE,
)

parameter_rows = [
    {"quantity": "valuation", "value": VALUE},
    {"quantity": "horizon T", "value": T},
    {"quantity": "trials", "value": N_TRIALS},
    {"quantity": "competitors", "value": N_COMPETITORS},
    {"quantity": "bid grid", "value": np.array2string(AVAILABLE_BIDS, precision=2)},
    {"quantity": "generous budget", "value": BUDGET_GENEROUS},
    {"quantity": "tight budget", "value": BUDGET_TIGHT},
]
markdown_table(parameter_rows, ["quantity", "value"])


## Clairvoyant Benchmarks

The benchmark is a clairvoyant LP that knows the true win probabilities. For a bid `b`, the expected utility is `(v - b) P(win)` and the expected cost is `b P(win)`. The LP maximizes expected utility while respecting `rho = B/T`.


In [ ]:
clairvoyant_rows = []
for scenario, budget in [("generous", BUDGET_GENEROUS), ("tight", BUDGET_TIGHT)]:
    env = SingleCampaignEnv(
        value=VALUE,
        budget=budget,
        T=T,
        available_bids=AVAILABLE_BIDS,
        n_competitors=N_COMPETITORS,
        seed=0,
    )
    gamma, opt_utility, exp_payment = compute_clairvoyant_single(
        env.available_bids,
        VALUE,
        budget / T,
        env.win_probabilities(),
    )
    active = [(float(b), float(p)) for b, p in zip(env.available_bids, gamma) if p > 1e-6]
    clairvoyant_rows.append({
        "scenario": scenario,
        "B": budget,
        "rho": budget / T,
        "opt utility/round": opt_utility,
        "expected cost/round": exp_payment,
        "active clairvoyant bids": active,
    })

markdown_table(
    clairvoyant_rows,
    ["scenario", "B", "rho", "opt utility/round", "expected cost/round", "active clairvoyant bids"],
    {"B": "{:.0f}", "rho": "{:.4f}", "opt utility/round": "{:.5f}", "expected cost/round": "{:.5f}"},
)


## Saved Results Summary

The following values are loaded from the pickled multi-trial results in `data/picklefiles/`. They summarize the final cumulative pseudo-regret and spend for each learner/scenario pair.


In [ ]:
results = {
    "generous / UCB1": (BUDGET_GENEROUS, load_pickle("req1_ucb1_generous_results.pkl")),
    "generous / UCB-like": (BUDGET_GENEROUS, load_pickle("req1_ucblike_generous_results.pkl")),
    "tight / UCB1": (BUDGET_TIGHT, load_pickle("req1_ucb1_tight_results.pkl")),
    "tight / UCB-like": (BUDGET_TIGHT, load_pickle("req1_ucblike_tight_results.pkl")),
}

summary_rows = []
for label, (budget, res) in results.items():
    final_regret = float(res["mean_regret"][-1])
    final_cost = float(res["mean_cumcost"][-1])
    summary_rows.append({
        "case": label,
        "trials": res["n_trials"],
        "final regret": final_regret,
        "avg regret": final_regret / len(res["mean_regret"]),
        "final cost": final_cost,
        "budget slack": budget - final_cost,
    })

markdown_table(
    summary_rows,
    ["case", "trials", "final regret", "avg regret", "final cost", "budget slack"],
    {"final regret": "{:.3f}", "avg regret": "{:.5f}", "final cost": "{:.3f}", "budget slack": "{:.3f}"},
)


## Final Req1 Figures

These are the cleaned final output figures restored from the exported image folder. The files in `outputs/req1/` were verified byte-for-byte against the uploaded originals.


In [ ]:
show_png("req1/01_cumulative_cost_final.png", width=950)
show_png("req1/02_cumulative_regret_final.png", width=950)
show_png("req1/03_average_regret_final.png", width=950)


## Reading the Results

For Requirement 1, the theory says that UCB-style methods should achieve no-regret behavior in the stochastic setting: cumulative regret may grow, but average regret should decrease over time.

The budget plots should be interpreted together with regret. A budget-aware learner is not necessarily trying to spend as fast as possible; it is trying to maximize utility while keeping cumulative cost compatible with the total budget. The tight-budget case is therefore the most important stress test: UCB1 can learn a high-utility bid while still being budget-unaware, while the UCB-like LP explicitly controls spend.


## Additional Explainability Figures

The figures below are stored separately from the final Req1 outputs. They are meant for slides/report explanations: which bids are selected, how often they win, how selected bids compare to competing bids, and how outcomes evolve over time.


In [ ]:
for rel in [
    "explainability/req1/01_cumulative_cost_with_pacing_tight.png",
    "explainability/req1/02_bid_frequency_tight.png",
    "explainability/req1/03_first500_bid_vs_competing_tight.png",
    "explainability/req1/04_win_rate_by_bid_tight_ucblike.png",
    "explainability/req1/05_rolling_outcomes_tight_ucblike.png",
    "explainability/req1/06_outcome_scatter_tight_ucblike.png",
    "explainability/req1/07_req1_tight_bid_win_dashboard_seed7.png",
    "explainability/req1/08_req1_bid_timeline_ucb1_vs_ucblike_seed7.png",
]:
    display(Markdown(f"### `{rel}`"))
    show_png(rel, width=900)


## Optional: Regenerate Req1 Outputs

Run this only if you intentionally want to overwrite the cleaned final images with freshly generated figures from the pipeline.


In [ ]:
# from utils.run_req1 import run_req1
# run_req1()
